In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_FILE = 'imu_challenge_dataset_ext.csv'
df = pd.read_csv(CSV_FILE, skiprows=1)
print(f'Rows: {len(df):,}   dt range: {df["time_s"].diff().describe()}')
df.head(3)

In [ ]:
# --- integrate gyro -> attitude angles (rad, then converted to deg) ---
dt = df['time_s'].diff().fillna(0).values  # time step per sample

roll_rad  = np.cumsum(df['gx_rad_s'].values * dt)  # rotation about X
pitch_rad = np.cumsum(df['gy_rad_s'].values * dt)  # rotation about Y
yaw_rad   = np.cumsum(df['gz_rad_s'].values * dt)  # rotation about Z

roll_deg  = np.degrees(roll_rad)
pitch_deg = np.degrees(pitch_rad)
yaw_deg   = np.degrees(yaw_rad)

t = df['time_s'].values

print(f'Roll  range: {roll_deg.min():.2f} … {roll_deg.max():.2f} deg')
print(f'Pitch range: {pitch_deg.min():.2f} … {pitch_deg.max():.2f} deg')
print(f'Yaw   range: {yaw_deg.min():.2f} … {yaw_deg.max():.2f} deg')

In [ ]:
# --- plot each angle on its own subplot ---
angles = [
    ('Roll  (gx)',  roll_deg,  'tab:blue'),
    ('Pitch (gy)', pitch_deg, 'tab:orange'),
    ('Yaw   (gz)', yaw_deg,   'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (label, data, color) in zip(axes, angles):
    ax.plot(t, data, linewidth=0.7, color=color)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('Gyro-Only Attitude Estimation (dead-reckoning)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gyro_attitude.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gyro_attitude.png')

In [ ]:
# --- combined overlay for quick comparison ---
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, roll_deg,  linewidth=0.7, label='Roll  (gx)', color='tab:blue')
ax.plot(t, pitch_deg, linewidth=0.7, label='Pitch (gy)', color='tab:orange')
ax.plot(t, yaw_deg,   linewidth=0.7, label='Yaw   (gz)', color='tab:green')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (deg)')
ax.set_title('Roll / Pitch / Yaw — Gyro Integration', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()